In [5]:
# Celda 1 - Librerias  [V0.7 - OpenCV Puro & GP Absoluto]
import cv2
import numpy as np
import time
import math
import threading
import collections
import customtkinter as ctk
from PIL import Image

# Librerias para Optimizador DEAP
import random
from deap import base, creator, tools, algorithms

ctk.set_appearance_mode('dark')
print('Celda 1 V0.6: Librerias base y DEAP listas.')

Celda 1 V0.6: Librerias base y DEAP listas.


In [6]:
# Celda 2 - Motor de vision con Contornos y Puntos
from pygrabber.dshow_graph import FilterGraph
import cv2
import numpy as np
import time
import math
import collections

def detectar_camaras_sistema():
    try:
        graph = FilterGraph()
        return [(i, n) for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar camaras: {e}')
        return []

RANGO_ROJO_1 = (np.array([  0, 100,  60]), np.array([ 12, 255, 255]))
RANGO_ROJO_2 = (np.array([168, 100,  60]), np.array([180, 255, 255]))
# Ajustamos el negro para no detectar sombras entre pelotas
RANGO_NEGRO  = (np.array([  0,   0,   0]), np.array([180, 255,  40]))
RANGO_BLANCO = (np.array([  0,   0, 170]), np.array([180,  45, 255]))

LIMITES_PELOTAS = {'Rojo': 10, 'Negro': 10, 'Blanco': 10}
MAX_PELOTAS_TOTAL = 10

def get_color_bgr(nombre):
    n = nombre.lower()
    if 'rojo'   in n or 'red'   in n: return (0,   0, 220)
    if 'blanco' in n or 'white' in n: return (200, 200, 200)
    if 'negro'  in n or 'black' in n: return (80,  80,  80)
    return (0, 220, 220)

def estimar_z(radio_px, frame_shape):
    frac = (math.pi * radio_px * radio_px) / (frame_shape[0] * frame_shape[1])
    return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)

class TrackerCirculo:
    def __init__(self, alpha=1.0, frames_conf=2, frames_perdida=5):
        self.alpha          = alpha
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.suave       = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, deteccion):
        if deteccion is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible = False
                self.suave   = None
            return tuple(int(round(v)) for v in self.suave) if self.visible else None
        
        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)
        
        if self.conteo_det < self.frames_conf: 
            return None
            
        if self.suave is None:
            self.suave   = tuple(float(v) for v in deteccion[:3])
            self.visible = True
            return tuple(int(round(v)) for v in deteccion[:3])
            
        self.visible = True
        return tuple(int(round(v)) for v in deteccion[:3])

def emparejar_detecciones(trackers, detecciones):
    asignaciones = [None] * len(trackers)
    if not detecciones: return asignaciones
    det_usadas = set()
    for i, tr in enumerate(trackers):
        if tr.suave is not None and tr.visible:
            mejor_det = None
            mejor_dist = float('inf')
            for j, d in enumerate(detecciones):
                if j in det_usadas: continue
                dist = math.hypot(tr.suave[0] - d[0], tr.suave[1] - d[1])
                if dist < mejor_dist and dist < 120:
                    mejor_dist = dist
                    mejor_det = j
            if mejor_det is not None:
                asignaciones[i] = detecciones[mejor_det]
                det_usadas.add(mejor_det)
    for j, d in enumerate(detecciones):
        if j not in det_usadas:
            for i, tr in enumerate(trackers):
                if asignaciones[i] is None and (tr.suave is None or not tr.visible):
                    asignaciones[i] = d
                    break
    return asignaciones

def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None, **kwargs):
    t_act       = time.time()
    hay_objetos = False
    fh, fw      = frame.shape[:2]
    detecciones_brutas = {'Rojo': [], 'Blanco': [], 'Negro': []}

    cx_scr, cy_scr = fw // 2, fh // 2
    mesh_size = 180
    
    cv2.rectangle(frame, (cx_scr - mesh_size, cy_scr - mesh_size), (cx_scr + mesh_size, cy_scr + mesh_size), (255, 255, 255), 1)
    cv2.line(frame, (cx_scr, cy_scr - mesh_size), (cx_scr, cy_scr + mesh_size), (255, 255, 255), 1)
    cv2.line(frame, (cx_scr - mesh_size, cy_scr), (cx_scr + mesh_size, cy_scr), (255, 255, 255), 1)
    cv2.circle(frame, (cx_scr, cy_scr), mesh_size, (255, 255, 255), 1)

    mask_centro = np.zeros((fh, fw), dtype=np.uint8)
    cv2.rectangle(mask_centro, (cx_scr - mesh_size, cy_scr - mesh_size), (cx_scr + mesh_size, cy_scr + mesh_size), 255, -1)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    mascaras = {
        'Rojo': cv2.bitwise_and(cv2.add(cv2.inRange(hsv, *RANGO_ROJO_1), cv2.inRange(hsv, *RANGO_ROJO_2)), mask_centro),
        'Negro': cv2.bitwise_and(cv2.inRange(hsv, *RANGO_NEGRO), mask_centro),
        'Blanco': cv2.bitwise_and(cv2.inRange(hsv, *RANGO_BLANCO), mask_centro)
    }

    for nombre, mascara in mascaras.items():
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, kernel, iterations=2)
        mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel, iterations=1)
        
        # Erodir agresivamente para separar pelotas juntas (especialmente negras)
        kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
        mascara_separada = cv2.erode(mascara, kernel_erode, iterations=2)
        
        cnts = cv2.findContours(mascara_separada, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contornos = cnts[0] if len(cnts) == 2 else cnts[1]
        
        for contorno in contornos:
            area = cv2.contourArea(contorno)
            if area > 100:  
                (cx, cy), radio = cv2.minEnclosingCircle(contorno)
                radio = radio + 10  # Compensar la erosion
                if 20 < radio < 140:
                    detecciones_brutas[nombre].append((int(cx), int(cy), int(radio)))

    total_pelotas_dibujadas = 0
    for nombre in ['Rojo', 'Blanco', 'Negro']:
        lista_detecciones = sorted(detecciones_brutas[nombre], key=lambda d: math.hypot(d[0]-cx_scr, d[1]-cy_scr))
        
        if total_pelotas_dibujadas >= MAX_PELOTAS_TOTAL:
            lista_detecciones = []
        else:
            cupo_restante = MAX_PELOTAS_TOTAL - total_pelotas_dibujadas
            lista_detecciones = lista_detecciones[:cupo_restante]
            
        asignaciones = emparejar_detecciones(trackers[nombre], lista_detecciones)
        
        for idx, det in enumerate(asignaciones):
            tr = trackers[nombre][idx]
            result = tr.actualizar(det)
            if result is None:
                temporizadores[nombre][idx] = 0.0
                continue
            
            if total_pelotas_dibujadas >= MAX_PELOTAS_TOTAL: break
                
            hay_objetos = True
            total_pelotas_dibujadas += 1
            cx, cy, r = result
            if temporizadores[nombre][idx] == 0.0: temporizadores[nombre][idx] = t_act
            
            color_bgr = get_color_bgr(nombre)
            x_norm = round(cx / fw * 2 - 1, 2)
            y_norm = round(1 - cy / fh * 2, 2)
            z_est  = estimar_z(r, frame.shape)
            
            cv2.circle(frame, (cx, cy), r, color_bgr, 3) 
            cv2.circle(frame, (cx, cy), 4, color_bgr, -1)
                
            lbl = f'{nombre} #{idx+1}'
            lbl_c = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'
            
            label_y = max(18, cy - r - 10)
            (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (cx - r, label_y - th - 4), (cx - r + tw + 4, label_y + 4), (20, 20, 20), -1)
            cv2.putText(frame, lbl, (cx - r + 2, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
            
            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (cx - r, cy2 - ch - 2), (cx - r + cw + 4, cy2 + 2), (20, 20, 20), -1)
            cv2.putText(frame, lbl_c, (cx - r + 2, cy2), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 210, 210), 1)
            
            if callback_objeto is not None: callback_objeto(lbl, x_norm, y_norm, z_est)
            
    return frame, hay_objetos
print('Celda 2 V0.10 lista.')


Celda 2 V0.7 lista.


In [7]:
# Celda 3 - UI y Logica de Puntos
import customtkinter as ctk
from PIL import Image
import threading

class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Deteccion de Carga - Puntos')
        self.geometry('1380x860')
        self.configure(fg_color='#1a1d2e')
        
        self.cap = None
        self.escaneando = False
        
        self.trackers = {
            'Rojo': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Rojo'])],
            'Blanco': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Blanco'])],
            'Negro': [TrackerCirculo() for _ in range(LIMITES_PELOTAS['Negro'])]
        }
        self.temporizadores = {
            'Rojo': [0.0]*LIMITES_PELOTAS['Rojo'],
            'Blanco': [0.0]*LIMITES_PELOTAS['Blanco'],
            'Negro': [0.0]*LIMITES_PELOTAS['Negro']
        }
        self.historial_objetos = set()
        self.puntos_actuales = 0
        self.limite_alcanzado = False
        
        self.cam_delay_ms = 33
        self._construir_ui()

    def _construir_ui(self):
        self.lbl_titulo = ctk.CTkLabel(self, text='CONFIGURACION DE CAMARA', font=('Helvetica', 22, 'bold'), text_color='#e8eaf0')
        self.lbl_titulo.pack(pady=(28, 12))
        self.panel_config = ctk.CTkFrame(self, fg_color='transparent')
        self.panel_config.pack(expand=True, fill='both')
        
        self.btn_detectar = ctk.CTkButton(self.panel_config, text='DETECTAR CAMARAS', command=self._accion_buscar)
        self.btn_detectar.pack(pady=10)
        self.frame_lista = ctk.CTkScrollableFrame(self.panel_config, width=700, height=150)
        self.frame_lista.pack(pady=5)
        self.indice_sel = ctk.StringVar(value='-1')
        self.btn_iniciar = ctk.CTkButton(self.panel_config, text='INICIAR DETECCION', state='disabled', command=self._iniciar_camara)
        self.btn_iniciar.pack(pady=40)
        
        self.panel_video = ctk.CTkFrame(self, fg_color='transparent')
        self.lbl_video = ctk.CTkLabel(self.panel_video, text='')
        self.lbl_video.pack(pady=10, padx=10)
        
        self.frame_controles = ctk.CTkFrame(self.panel_video, fg_color='transparent')
        self.frame_controles.pack(pady=8)
        
        self.btn_escaneo = ctk.CTkButton(self.frame_controles, text='Iniciar Escaneo', command=self._toggle_escaneo)
        self.btn_escaneo.grid(row=0, column=0, padx=10)
        
        self.btn_limpiar = ctk.CTkButton(self.frame_controles, text='Limpiar Todo', command=self._limpiar_todo)
        self.btn_limpiar.grid(row=0, column=1, padx=10)
        
        self.btn_detener = ctk.CTkButton(self.frame_controles, text='Dejar de Escanear', fg_color='#e74c3c', hover_color='#c0392b', command=self._detener_camara)
        self.btn_detener.grid(row=0, column=2, padx=10)
        
        self.lbl_puntos = ctk.CTkLabel(self.frame_controles, text='Puntos: 0 / 10 MAX', font=('Helvetica', 16, 'bold'), text_color='#3498db')
        self.lbl_puntos.grid(row=0, column=3, padx=20)
        
        self.lbl_gp_status = ctk.CTkLabel(self.panel_video, text='', text_color='#f1c40f', font=('Helvetica', 14, 'bold'))
        self.lbl_gp_status.pack(pady=5)

        self.lista_scroll = ctk.CTkScrollableFrame(self.panel_video, width=300)
        self.lista_scroll.pack(side='right', fill='y', padx=10, pady=10)

    def _accion_buscar(self):
        for w in self.frame_lista.winfo_children(): w.destroy()
        camaras = detectar_camaras_sistema()
        if camaras:
            for idx, etq in camaras:
                ctk.CTkRadioButton(self.frame_lista, text=etq, variable=self.indice_sel, value=str(idx)).pack(anchor='w', pady=5)
            self.indice_sel.set(str(camaras[0][0]))
            self.btn_iniciar.configure(state='normal')

    def _iniciar_camara(self):
        idx = int(self.indice_sel.get())
        if idx == -1: return
        self.panel_config.pack_forget()
        self.panel_video.pack(expand=True, fill='both')
        self.lbl_titulo.configure(text='MONITOR EN VIVO')
        
        self.cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
        if not self.cap.isOpened() or not self.cap.read()[0]:
            self.cap = cv2.VideoCapture(idx)
            
        if not self.cap.isOpened():
            self.lbl_video.configure(text='ERROR', text_color='red')
            return
            
        self.lbl_video.configure(text='')
        self.escaneando = False
        self._loop_video()

    def _detener_camara(self):
        self.escaneando = False
        if self.cap: self.cap.release()
        self.panel_video.pack_forget()
        self.panel_config.pack(expand=True, fill='both')

    def _toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando: self.btn_escaneo.configure(text='Pausar Escaneo', fg_color='#d35400')
        else: self.btn_escaneo.configure(text='Reanudar Escaneo', fg_color='#f39c12')

    def _loop_video(self):
        if not self.cap or not self.cap.isOpened(): return
        
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            if self.escaneando and not self.limite_alcanzado:
                frame, _ = procesar_frame_vision(
                    frame, self.temporizadores, self.trackers, 
                    callback_objeto=self.on_objeto_detectado
                )
            img = ctk.CTkImage(light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), size=(920, 630))
            self.lbl_video.configure(image=img)
        
        self.after(self.cam_delay_ms, self._loop_video)

    def on_objeto_detectado(self, lbl_nombre, x, y, z):
        if self.limite_alcanzado: return
        
        if lbl_nombre not in self.historial_objetos:
            pts = 0
            if 'Negro' in lbl_nombre: pts = 5
            elif 'Blanco' in lbl_nombre: pts = 3
            elif 'Rojo' in lbl_nombre: pts = 1
            
            if self.puntos_actuales + pts > 10:
                self.historial_objetos.add(lbl_nombre)
                self.puntos_actuales += pts
                self.limite_alcanzado = True
                self.escaneando = False
                ctk.CTkLabel(self.lista_scroll, text=f"{lbl_nombre} (+{pts} pts) -> ¡LIMITE!").pack()
                self.lbl_puntos.configure(text=f'Puntos: {self.puntos_actuales} / 10 MAX', text_color='red')
                self.btn_escaneo.configure(text='Límite Alcanzado', state='disabled', fg_color='#7f8c8d')
                self.lbl_gp_status.configure(text=f'¡CANASTA LLENA! ({self.puntos_actuales} pts detectados)', text_color='red')
                return

            self.historial_objetos.add(lbl_nombre)
            self.puntos_actuales += pts
            ctk.CTkLabel(self.lista_scroll, text=f"{lbl_nombre} (+{pts} pts)").pack()
            self.lbl_puntos.configure(text=f'Puntos: {self.puntos_actuales} / 10 MAX')
            
            if self.puntos_actuales == 10:
                self.limite_alcanzado = True
                self.escaneando = False
                self.btn_escaneo.configure(text='Límite Alcanzado', state='disabled', fg_color='#7f8c8d')
                self.lbl_gp_status.configure(text=f'¡CANASTA LLENA! (10 pts)', text_color='red')
                self.lbl_puntos.configure(text_color='red')

    def _limpiar_todo(self):
        for w in self.lista_scroll.winfo_children(): w.destroy()
        self.historial_objetos.clear()
        self.puntos_actuales = 0
        self.limite_alcanzado = False
        self.lbl_puntos.configure(text='Puntos: 0 / 10 MAX', text_color='#3498db')
        self.lbl_gp_status.configure(text='')
        self.btn_escaneo.configure(state='normal', text='Iniciar Escaneo', fg_color='#1f6aa5')
        self.escaneando = False
        for c in self.temporizadores: self.temporizadores[c] = [0.0]*LIMITES_PELOTAS[c]
        for c in self.trackers: 
            for tr in self.trackers[c]: tr.reiniciar()

print('Celda 3 V0.10 lista.')


Celda 3 V0.6 lista.


In [8]:
# Celda 4 - Punto de entrada  [V0.6]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()